# Evaluation of Formulas from First Order Logic 

In this notebook, we show how formulas from *first order logic* can be evaluated in TypeScript.

## The Axioms of Group Theory

To have a nontrivial example of formulas, we use the formulas from [group theory](https://en.wikipedia.org/wiki/Group_theory).
A group is defined as a triple $ \langle G, \mathrm{e}, \circ \rangle $
where:
- $G$ is a non-empty set,
- $\mathrm{e}$ is an element from $G$, and
- $\circ:G \times G \rightarrow G$ is a binary function on $G$.

Furthermore, the following axioms have to be satisfied:
  * $\forall x: \mathrm{e} \circ x = x$
  * $\forall x: \exists y: y \circ x = \mathrm{e}$
  * $\forall x: \forall y: \forall z: (x \circ y) \circ z = x \circ (y \circ z)$

A group is *commutative* if, additionally, the following formula is satisfied:
  * $\forall x: \forall y: x \circ y = y \circ x$

We import the definition of `AST` and various other types from our new parser.
An object of type `AST` is an *abstract syntax tree* that is represented as a nested list.  The first element of these
lists is always a string representing an *operator*, a *predicate symbol*, or a *function symbol*.

In [7]:
import { parseFormula as parse, Formula, Term, Variable } from './FOL-Parser';

We also need the library `recursive-set`.

In [2]:
import { Tuple, RecursiveMap as Map, RecursiveSet as Set, Value } from 'recursive-set';

In [3]:
function set<T extends Value>(...elements: T[]): Set<T> {
    return new Set(...elements);
}
function tpl<T extends Value[]>(...elements: T): Tuple<T> {
    return new Tuple(...elements);
}

`map` takes a list of *key-value* pairs and creates a `Map`.  For example,
```
      map([[[0, 0], 0], 
           [[0, 1], 1], 
           [[1, 0], 1], 
           [[1, 1], 0]])
```
creates the `Map`
$$ \bigl\{ \langle 0, 0\rangle \mapsto 0,\;
           \langle 0, 1\rangle \mapsto 1,\;
           \langle 1, 0\rangle \mapsto 1,\;
           \langle 1, 1\rangle \mapsto 0
   \bigr\}. 
$$

In [4]:
function map<K extends Value, V extends Value>(entries?: [any, V][]): Map<K, V> {
    const m = new Map<K, V>();    
    if (entries) {
        for (const [k, v] of entries) {
            // If the key is a standard array (like [0, 0]), convert it to a Tuple.
            // Otherwise, use it as-is (e.g., for string keys).
            const key = Array.isArray(k) ? tpl(...k) as unknown as K : k as K;
            m.set(key, v);
        }
    }
    return m;
}

Our new parser distinguishes between variables and function/predicate symbols as follows:
- *Variables* start with an **upper case** letter (e.g., `X`, `Y`).
- *Functions* and *predicates* start with a **lower case** letter (e.g., `e`, `p`).
- We can use the arithmetic operators `+`, `-`, `*`, `/`, `%`, `**` as infix operators.
- We can use the relational operators `<`, `>`, `=`, `≤`, `≥` as infix operators.
- The logical symbols `⊤`, `⊥`, `∧`, `∨`, `¬`, `→`, `↔`, `∀`, `∃` are supported with the usual precedences.

  The quantifiers `∀`, `∃` bind stronger than `∧` and `∨`.

  **Note** that `∧` binds stronger than `∨`.  However, in order to avoid confusion 
  we will not mix `∧` and `∨` in this lesson. 

We represent the symbols from group theory as follows:
- The neutral element "$\mathrm{e}$" is the 0-ary function `e`.
- The operator "$\circ$" is represented by the arithmetic operator `*`.
- Equality "$=$" is natively supported by the parser as `=`.

Then the formulas of group theory can be written as shown below.

In [5]:
const G1 = '∀X: e * X = X';
const G2 = '∀X: ∃Y: Y * X = e';
const G3 = '∀X: ∀Y: ∀Z: (X * Y) * Z = X * (Y * Z)';
const G4 = '∀X: ∀Y: X * Y = Y * X';

In [8]:
const F1 = parse(G1);
console.dir(F1, { depth: null });

[
  '∀',
  'X',
  [ '=', [ '*', [ 'e' ], 'X' ], 'X' ]
]


In [9]:
const F2 = parse(G2);
console.dir(F2, { depth: null });

[
  '∀',
  'X',
  [ '∃', 'Y', [ '=', [ '*', 'Y', 'X' ], [ 'e' ] ] ]
]


In [10]:
const F3 = parse(G3);
console.dir(F3, { depth: null });

[
  '∀',
  'X',
  [
    '∀',
    'Y',
    [
      '∀',
      'Z',
      [
        '=',
        [ '*', [ '*', 'X', 'Y' ], 'Z' ],
        [ '*', 'X', [ '*', 'Y', 'Z' ] ]
      ]
    ]
  ]
]


In [11]:
const F4 = parse(G4);
console.dir(F4, { depth: null });

[
  '∀',
  'X',
  [ '∀', 'Y', [ '=', [ '*', 'X', 'Y' ], [ '*', 'Y', 'X' ] ] ]
]


## A Structure for Group Theory

We use `recursive-set` types to map tuples of arguments to their function results or predicate truth values.
To simplify matters, we assume that the universe is a set of numbers.

In [12]:
type Universe                = Set<number>;
type FunctionInterpretation  = Map<Tuple<number[]>, number>;
type PredicateInterpretation = Set<Tuple<number[]>>;

Assume a signature 
$$ \Sigma = \langle \mathcal{V}, \mathcal{F}, \mathcal{P}, \texttt{arity} \rangle $$
is given. A *$\Sigma$-structure* $\mathcal{S}$ is a
pair $\langle \mathcal{U}, \mathcal{J} \rangle$, such that the following holds:
 * $\mathcal{U}$ is a non-empty set. This set is also called the
   *universe* of the $\Sigma$-structure. This universe contains the *values*.
 * $\mathcal{J}$ is the *interpretation* of the function and predicate symbols.

   Formally, we define $\mathcal{J}$ as a mapping with the following properties:
   + Every function symbol $c \in \mathcal{F}$ with $\texttt{arity}(c) = 0$ is mapped to an element
     of $\mathcal{U}$, i.e.we have $\mathcal{J}(c) \in \mathcal{U}$.  
     
     Instead of writing $\mathcal{J}(c)$ we will write $c^{\mathcal{J}}$.
   + Every function symbol $f \in \mathcal{F}$ with $\texttt{arity}(f) = m$ and $m > 0$ is mapped to
     an $m$-ary function
     $$ f^\mathcal{J}\colon \mathcal{U}^m \rightarrow \mathcal{U}$$
     that maps $m$-tuples of the universe $\mathcal{U}$ into the universe $\mathcal{U}$.
   + Every predicate symbol $p \in \mathcal{P}$ with $\texttt{arity}(p) = 0$ is mapped to
     the set $\mathbb{B}$ of truth values, i.e. $p^{\mathcal{J}} \in \{ \mathtt{true}, \mathtt{false}\}$.
   + Every predicate symbol $p \in \mathcal{P}$ with $\texttt{arity}(p) = n$ such that $n > 0$ is mapped to
     a subset 
     $$ p^\mathcal{J} \subseteq \mathcal{U}^n. $$

     The idea is that an atomic formula of the form $p(t_1, \cdots, t_n)$
     is interpreted as $\texttt{true}$ exactly if and only if the interpretation of the tuple
     $\langle t_1, \cdots, t_n \rangle$ is an element of the set $p^\mathcal{J}$.
   + If the symbol `=` is a member of the set of predicate symbols $\mathcal{P}$, then the
     interpretation of  the equality symbol `=` has to be *canonical*, i.e. we must have 
     $$ =^\mathcal{J} \;=\; \bigl\{ \langle u, u \rangle \mid u \in \mathcal{U} \bigr\}. $$
     
A $\Sigma$-structure $\mathcal{S} = \langle \mathcal{U}, \mathcal{J} \rangle$ is represented as a **TypeScript** *object* that has the 
following properties:
* `universe` is the universe $\mathcal{U}$,
* `functions` is a `Map` that maps every function symbol $f$ to a `Map` $f^{\mathcal{J}}$. In turn,
  $f^{\mathcal{J}}$ maps tuples of numbers to numbers.
* `predicates` is a `Map` that maps every $n$-ary predicate symbol $p$ to a `Set` $p^{\mathcal{J}}$. In turn,
  $p^{\mathcal{J}}$ is a set of tuples of length $n$.

In [13]:
interface Structure {
    universe:   Universe;
    functions:  Map<string, FunctionInterpretation>;
    predicates: Map<string, PredicateInterpretation>;
}

Assume a signature 
$$\Sigma = \langle \mathcal{V}, \mathcal{F}, \mathcal{P}, \texttt{arity} \rangle$$
is given.  Furthermore,  $\mathcal{S} = \langle \mathcal{U}, \mathcal{J} \rangle$ is a  $\Sigma$-structure.
Then a function of the form 
$$ \mathcal{I}: \mathcal{V} \rightarrow \mathcal{U} $$
is an *$\mathcal{S}$ variable assignment*.

This function is represented as a `Map` mapping variable names to numbers.

In [14]:
type VariableAssignment = Map<string, number>;

We define the universe `U` as the set $\{0, 1\}$:

In [15]:
const U: Universe = set(0, 1);

We define interpretations for the constant `e`  and the binary function symbol `*`.

In [16]:
const NeutralElement: FunctionInterpretation = map([[[], 0]]);
NeutralElement

RecursiveMap(1) {
  () => 0
}


In [17]:
const Product: FunctionInterpretation = 
      map([[[0, 0], 0], 
           [[0, 1], 1], 
           [[1, 0], 1], 
           [[1, 1], 0]])
Product

RecursiveMap(4) {
  (0, 0) => 0,
  (0, 1) => 1,
  (1, 0) => 1,
  (1, 1) => 0
}


The interpretation of `=` is the set $\bigl\{ \langle x, x \rangle \mid x \in \mathcal{U} \bigr\}$.

In [18]:
const Identity = U.map(x => tpl(x, x));
Identity

{(0, 0), (1, 1)}


We proceed to define the interpretation $\mathcal{J}$.  This interpretation is split in the interpretation of the function symbols and the interpretation of the predicate symbols.

In [19]:
const functions = map<string, FunctionInterpretation>();
functions.set('e', NeutralElement);
functions.set('*', Product);
functions

RecursiveMap(2) {
  * => RecursiveMap(4) {
  (0, 0) => 0,
  (0, 1) => 1,
  (1, 0) => 1,
  (1, 1) => 0
},
  e => RecursiveMap(1) {
  () => 0
}
}


In [20]:
const predicates = map<string, PredicateInterpretation>();
predicates.set('=', Identity);
predicates

RecursiveMap(1) {
  = => {(0, 0), (1, 1)}
}


In [21]:
const S: Structure = { 
    universe:   U, 
    functions:  functions, 
    predicates: predicates 
};
S

{
  universe: {0, 1},
  functions: RecursiveMap(2) {
    * => RecursiveMap(4) {
    (0, 0) => 0,
    (0, 1) => 1,
    (1, 0) => 1,
    (1, 1) => 0
  },
    e => RecursiveMap(1) {
    () => 0
  }
  },
  predicates: RecursiveMap(1) {
    = => {(0, 0), (1, 1)}
  }
}


We define a variable assignment $\mathcal{I}$.  The concrete definition given here is arbitrary, because the axioms for group theory contain no free variables.

In [22]:
const I: VariableAssignment = map([['X', 0], ['Y', 1], ['Z', 0]]);
I

RecursiveMap(3) {
  X => 0,
  Y => 1,
  Z => 0
}


## Evaluation Functions

If $\Sigma = \langle \mathcal{V}, \mathcal{F}, \mathcal{P}, \texttt{arity} \rangle$ is a signature
$\mathcal{S} = \langle\mathcal{U},\mathcal{J}\rangle$ is a  $\Sigma$-structure and $\mathcal{I}$ is an  $\mathcal{S}$
variable assignment, then for every term $t$ the *value $\mathcal{S}(\mathcal{I}, t)$*
is defined by induction on $t$:
 + If $x \in \mathcal{V}$ we define:
   $$\mathcal{S}(\mathcal{I}, x) := \mathcal{I}(x). $$
 + If $c \in \mathcal{F}$ and $\texttt{arity}(c) = 0$, then 
   $$\mathcal{S}(\mathcal{I}, c) := c^\mathcal{J}. $$
 + If $t$ is a term of the form $f(t_1,\cdots,t_n)$, then we define 
   $$\mathcal{S}\bigl(\mathcal{I}, f(t_1,\cdots,t_n)\bigr) := 
                           f^\mathcal{J}\bigl( \mathcal{S}(\mathcal{I}, t_1), \cdots, \mathcal{S}(\mathcal{I}, t_n) \bigr). $$

The function `evalTerm`$(t, \mathcal{S}, \mathcal{I})$ takes
 - a term $t$,
 - a structure $\mathcal{S}$, and
 - a variable assignment $\mathcal{I}$,
and computes the value $\mathcal{S}(\mathcal{I}, t)$, i.e. it evaluates the term $t$ in the structure $\mathcal{S}$ with the variable assignment $\mathcal{I}$. 

In [23]:
function evalTerm(t: Term, S: Structure, I: VariableAssignment): number {
    if (typeof t == 'string') { return I.get(t); }
    const [fct, ...args] = t;
    const argValues = args.map(arg => evalTerm(arg, S, I));
    const fJ = S.functions.get(fct);
    return fJ.get(tpl(...argValues));
}

The function `modify`$(\mathcal{I}, x, c)$ takes
 - a variable assignment $\mathcal{I}$,
 - a variable $x$, and 
 - a value $c$, 
as inputs and computes the modified variable assignement $\mathcal{I}[x/c]$.  Here,
$\mathcal{I}[x/c]$ denotes the variable assignment
that maps the variable  $x$ to the value $c$ and that otherwise agrees with $\mathcal{I}$
$$\mathcal{I}[x/c](y) := \left\{
    \begin{array}{ll}
    c               & \mbox{if}\; y = x;  \\
    \mathcal{I}(y)  & \mbox{otherwise}.          \\
    \end{array}
    \right.
$$

In [24]:
function modify(I: VariableAssignment, x: string, c: number): VariableAssignment {
    const Is = I.mutableCopy();
    Is.set(x, c);
    return Is;
}

The function `evalFormula`$(F, \mathcal{S}, \mathcal{I})$ takes
 - a formula $F$,
 - a structure $\mathcal{S}$, and
 - a variable assignment $\mathcal{I}$
as inputs and computes the value $\mathcal{S}(\mathcal{I}, F)$, i.e. it evaluates the formula $F$ in the structure $\mathcal{S}$ with the variable assignment $\mathcal{I}$.
 
Let $\mathcal{S} = \langle \mathcal{U}, \mathcal{J} \rangle$ be a $\Sigma$-structure and $\mathcal{I}$ a $\mathcal{S}$-variable 
assignment.  Then the value $\mathcal{S}(\mathcal{I},F)$ is defined by induction:
  * $\mathcal{S}(\mathcal{I},\top) := \mathtt{true}$ and $\mathcal{S}(\mathcal{I},\bot) := \mathtt{false}$.
  * If $p$ is a propositional variable, then 
    $$\mathcal{S}(\mathcal{I}, p) := p^\mathcal{J}.$$
  * $\mathcal{S}\bigl(\mathcal{I}, p(t_1,\cdots,t_n)\bigr) := 
          \Bigl(\bigl\langle \mathcal{S}(\mathcal{I}, t_1), \cdots, \mathcal{S}(\mathcal{I}, t_n) \bigr\rangle \in p^\mathcal{J}\Bigr)$.
  * $\mathcal{S}(\mathcal{I}, \neg F) := \require{enclose} \enclose{circle}{\neg}\bigl(\mathcal{S}(\mathcal{I}, F)\bigr)$.
  * $\mathcal{S}(\mathcal{I}, F \wedge G) := \require{enclose} \enclose{circle}\wedge\bigl(\mathcal{S}(\mathcal{I}, F), \mathcal{S}(\mathcal{I}, G)\bigr)$.
  * $\mathcal{S}(\mathcal{I}, F \vee G) := \require{enclose} \enclose{circle}\vee\bigl(\mathcal{S}(\mathcal{I}, F), \mathcal{S}(\mathcal{I}, G)\bigr)$.
  * $\mathcal{S}(\mathcal{I}, F \rightarrow G) := \require{enclose} \enclose{circle}\rightarrow\bigl(\mathcal{S}(\mathcal{I}, F), \mathcal{S}(\mathcal{I}, G)\bigr)$.
  * $\mathcal{S}(\mathcal{I}, F \leftrightarrow G) := \require{enclose} \enclose{circle}\leftrightarrow\bigl(\mathcal{S}(\mathcal{I}, F), \mathcal{S}(\mathcal{I}, G)\bigr)$.
  * $\mathcal{S}\bigl(\mathcal{I}, \forall x\colon F\bigr) := \left\{
      \begin{array}{ll}
         \mathtt{true}  & \mbox{if}\; \mathcal{S}(\mathcal{I}[x/c], F) = \mathtt{true}\quad \mbox{holds for all}\; c\in \mathcal{U}; \\
         \mathtt{false} & \mbox{otherwise}.
      \end{array}
      \right.$
   * $\mathcal{S}\bigl(\mathcal{I}, \exists x \colon F\bigr) := \left\{
      \begin{array}{ll}
         \mathtt{true}  & \mbox{if}\; \mathcal{S}(\mathcal{I}[x/c], F) = \mathtt{true}\quad \mbox{holds for some}\; c\in \mathcal{U}; \\
         \mathtt{false} & \mbox{otherwise}.
      \end{array}
      \right.$

In [26]:
function evalFormula(F: Formula, S: Structure, I: VariableAssignment): boolean {
    switch (F[0]) {
        case '⊤': return true;
        case '⊥': return false;
        case '¬': return !evalFormula(F[1] as Formula, S, I);
        case '∧':  
        case '∨':  
        case '→':  
        case '↔': 
            const [op, H, K] = F as ['∧'|'∨'|'→'|'↔', Formula, Formula];
            const [Hs, Ks] = [H, K].map(c => evalFormula(c, S, I));
            switch (op) {
                case '∧': return Hs && Ks;
                case '∨': return Hs || Ks;
                case '→': return Hs <= Ks;
                case '↔': return Hs == Ks;
            }
        case '∀':  case '∃': 
            const [Q, x, G] = F as ['∀' | '∃', Variable, Formula];
            const evalAt = c => evalFormula(G, S, modify(I, x, c));
            switch (Q) {
                case '∀': return S.universe.every(evalAt);
                case '∃': return S.universe.some (evalAt);
            }
        default: 
            const [p, ...args] = F;
            const argValues = args.map(arg => evalTerm(arg, S, I));
            const pJ = S.predicates.get(p);            
            return pJ.has(tpl(...argValues));
}}

## Checking whether $\mathcal{S}$ is a Group

In [27]:
S

{
  universe: {0, 1},
  functions: RecursiveMap(2) {
    * => RecursiveMap(4) {
    (0, 0) => 0,
    (0, 1) => 1,
    (1, 0) => 1,
    (1, 1) => 0
  },
    e => RecursiveMap(1) {
    () => 0
  }
  },
  predicates: RecursiveMap(1) {
    = => {(0, 0), (1, 1)}
  }
}


In [28]:
console.log(`evalFormula(G1) = ${evalFormula(F1, S, I)}`);
console.log(`evalFormula(G2) = ${evalFormula(F2, S, I)}`);
console.log(`evalFormula(G3) = ${evalFormula(F3, S, I)}`);
console.log(`evalFormula(G4) = ${evalFormula(F4, S, I)}`);

evalFormula(G1) = true
evalFormula(G2) = true
evalFormula(G3) = true
evalFormula(G4) = true


## Another Example

Let's show that the formula $\forall X: \exists Y: p(X,Y) \rightarrow \exists Y: \forall X: p(X,Y)$ is not universally valid.

In [29]:
const F_invalid = parse('∀X: ∃Y: p(X, Y) → ∃Y: ∀X: p(X, Y)');

In [30]:
const U2: Universe = set(0, 1);

const pJ: PredicateInterpretation = set(tpl(0, 1), tpl(1, 0));
const predicates2 = map<string, PredicateInterpretation>();
predicates2.set('p', pJ);

const S2: Structure = { 
    universe:   U2, 
    functions:  map(), 
    predicates: predicates2 
};
S2

{
  universe: {0, 1},
  functions: RecursiveMap(0) {},
  predicates: RecursiveMap(1) {
    p => {(0, 1), (1, 0)}
  }
}


In [31]:
const I2 = map<string, number>([['X', 0],['Y', 0]]);
I2

RecursiveMap(2) {
  X => 0,
  Y => 0
}


In [32]:
console.log(`Result: ${evalFormula(F_invalid, S2, I2)}`);

Result: false
